[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Sessions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the classes, `hero_engine`, and `scratch/heroes.db` with the
eight heroes and three teams loaded. Run it first. The tasks change the same database in order, and
the last cell removes the scratch folder.


In [1]:
import logging
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, insert, inspect, text
from sqlalchemy.exc import IntegrityError, PendingRollbackError
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with engine.connect() as connection:
    print("sqlmodel", sqlmodel.__version__, "|",
          connection.execute(text("select count(*) from hero")).scalar(), "heroes in",
          connection.execute(text("select count(*) from team")).scalar(), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


**1.** A team, and the id that arrives at the commit.


In [2]:
with Session(engine) as session:
    council = Team(name="Grand Council", headquarters="Grand Hall")
    session.add(council)
    print("before the commit:", council.id)
    session.commit()
    print("after the commit :", council.id)


before the commit: None
after the commit : 4


The database gave the row its id, and reading the attribute after the commit fetched it back.


**2.** Two heroes, one commit.


In [3]:
with Session(engine) as session:
    session.add(Hero(name="Lady Ice", secret_name="Kara Frost", age=29, team_id=1))
    session.add(Hero(name="Night Owl", secret_name="Bruce Kent", age=41, team_id=2))
    engine.echo = True
    session.commit()
    engine.echo = False
# Two INSERTs, one for each hero, each ending in RETURNING id, with one COMMIT around both.


    BEGIN (implicit)
    INSERT INTO hero (name, secret_name, age, team_id) VALUES (?, ?, ?, ?) RETURNING id
    values: ('Lady Ice', 'Kara Frost', 29, 1)
    INSERT INTO hero (name, secret_name, age, team_id) VALUES (?, ?, ?, ?) RETURNING id
    values: ('Night Owl', 'Bruce Kent', 41, 2)
    COMMIT


Two objects, two statements, one transaction. Each `INSERT` ends in `RETURNING id`, which is how the
session learns what the database gave each row. The single hero in the notebook's first example got
an `INSERT` with no `RETURNING`: with one row there is nothing to match up, and SQLite reports the id
of the last row it wrote.


**3.** What a commit does to an object it holds.


In [4]:
with Session(engine) as session:
    hero = session.get(Hero, 2)
    hero.age = 17
    print("before the commit:", sorted(inspect(hero).unloaded))
    session.commit()
    print("after the commit :", sorted(inspect(hero).unloaded))
# Before, every value was loaded, since get had just read the row. The commit expired all of them,
# so the next read of any one fetches the whole row again.


before the commit: []
after the commit : ['age', 'id', 'name', 'secret_name', 'team_id']


The change was written first: the commit sent the `UPDATE`, then expired the object, which is why
nothing is left loaded.


**4.** A hero that outlives its session.


In [5]:
def hero_by_id(engine, hero_id):
    """One hero, loaded and kept readable after the session closes."""
    with Session(engine) as session:
        hero = session.get(Hero, hero_id)
        session.refresh(hero)                                       # every column loaded, while there is a session
        return hero


rusty = hero_by_id(engine, 3)
print(fields(rusty))


{'id': 3, 'name': 'Rusty-Man', 'secret_name': 'Tommy Sharp', 'age': 48, 'team_id': 1}


`refresh` loads every column, and nothing expires them afterwards, so the object is readable with no
session at all. It is still a detached object: a relationship that was never loaded would still
raise, which the **Relationships** notebook shows.


**5.** Every Preventer moved to the Z-Force.


In [6]:
with Session(engine) as session:
    moving = session.exec(select(Hero).where(Hero.team_id == 1)).all()
    for hero in moving:
        hero.team_id = 2
    print("changed:", len(session.dirty), "heroes:", sorted(hero.name for hero in moving))
    session.commit()

with Session(engine) as session:
    print("on the Preventers now:", len(session.exec(select(Hero).where(Hero.team_id == 1)).all()))
    print("on the Z-Force now   :", len(session.exec(select(Hero).where(Hero.team_id == 2)).all()))


changed: 5 heroes: ['Captain North America', 'Lady Ice', 'Rusty-Man', 'Spider-Boy', 'Tarantula']
on the Preventers now: 0
on the Z-Force now   : 9


The session had loaded all five, so it knew which ones changed and sent an `UPDATE` for each. The
second session read the rows again and found the Preventers empty.


**6.** Two deletes, one of which the database refuses.


In [7]:
with Session(engine) as session:
    wakaland = session.get(Team, 3)
    name = wakaland.name                                            # read it before the row is gone
    session.delete(wakaland)
    session.commit()
    print("deleted:", name, "| teams left:", len(session.exec(select(Team)).all()))

    session.delete(session.get(Team, 2))
    try:
        session.commit()
    except IntegrityError as error:
        print("refused:", str(error).splitlines()[0])
    session.rollback()
    print("still usable:", session.get(Team, 2).name)


deleted: Wakaland Guard | teams left: 3
refused: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
still usable: Z-Force


The team with no heroes went; the one every hero now points at did not. The rollback undid the
delete and left the session working, which is what any caught `IntegrityError` has to do.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Sessions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/03-sessions.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
